In [2]:
from tool_server.utils.utils import *


In [ ]:
input_data_path = "share_data/datasets/web/gemini_test_format_conversation.jsonl"
web_train_data = process_jsonl(input_data_path)

In [4]:
len(web_train_data)

1139

In [5]:
for item in web_train_data:
    for conv in item:
        contents = conv["content"]
        for content in contents:
            if content["type"] == "image_url":
                image_path = content["image_url"]["url"]
                assert os.path.exists(image_path), image_path

In [ ]:
import base64
import io
from IPython.display import display, HTML
from PIL import Image

import matplotlib.pyplot as plt

def visualize_conversation(item_idx=0, max_width=600):
    """
    Visualize a conversation from the dataset with text and images.
    
    Args:
        item_idx: Index of the conversation to visualize
        max_width: Maximum width for displayed images
    """
    item = web_train_data[item_idx]
    
    html_content = "<div style='font-family: Arial, sans-serif; max-width: 800px; margin: 0 auto;'>"
    
    for i, conv in enumerate(item):
        role = conv["role"]
        
        # Add role header with different styling based on role
        if role == "system":
            html_content += f"<div style='background-color: #f0f0f0; padding: 10px; border-left: 5px solid #888; margin-bottom: 15px;'>"
            html_content += f"<h3 style='margin: 0 0 10px 0; color: #000;'>System</h3>"
        elif role == "user":
            html_content += f"<div style='background-color: #e8f4ff; padding: 10px; border-left: 5px solid #4a86e8; margin-bottom: 15px;'>"
            html_content += f"<h3 style='margin: 0 0 10px 0; color: #000;'>User</h3>"
        elif role == "assistant":
            html_content += f"<div style='background-color: #f0f7e8; padding: 10px; border-left: 5px solid #6aa84f; margin-bottom: 15px;'>"
            html_content += f"<h3 style='margin: 0 0 10px 0; color: #000;'>Assistant</h3>"
        
        # Process content
        for content in conv["content"]:
            if content["type"] == "text":
                # Format the text with proper line breaks and paragraphs
                text = content["text"].replace('\n', '<br>')
                html_content += f"<p style='margin: 5px 0; color: #000;'>{text}</p>"
            
            elif content["type"] == "image_url":
                image_path = content["image_url"]["url"]
                try:
                    # Open image and resize if needed
                    img = Image.open(image_path)
                    width, height = img.size
                    
                    # Calculate new dimensions if width exceeds max_width
                    if width > max_width:
                        ratio = max_width / width
                        new_width = max_width
                        new_height = int(height * ratio)
                    else:
                        new_width = width
                        new_height = height
                    
                    # Convert image to base64 for embedding in HTML
                    buffer = io.BytesIO()
                    img.save(buffer, format="PNG")
                    img_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
                    
                    html_content += f"<div style='margin: 10px 0;'>"
                    html_content += f"<img src='data:image/png;base64,{img_str}' width='{new_width}' height='{new_height}' style='border: 1px solid #ddd; box-shadow: 2px 2px 5px #ccc;'>"
                    html_content += f"<div style='font-size: 12px; color: #000; margin-top: 3px;'>Image from: {image_path}</div>"
                    html_content += "</div>"
                except Exception as e:
                    html_content += f"<p style='color: red;'>Failed to load image: {image_path}<br>Error: {str(e)}</p>"
        
        html_content += "</div>"
    
    html_content += "</div>"
    
    # Display the HTML content
    display(HTML(html_content))

# Example usage: Visualize the first conversation
visualize_conversation(0)

# Function to browse through conversations
def browse_conversations(start_idx=0, count=3):
    """
    Browse through multiple conversations
    
    Args:
        start_idx: Starting index
        count: Number of conversations to display
    """
    end_idx = min(start_idx + count, len(web_train_data))
    
    for idx in range(start_idx, end_idx):
        print(f"\n{'='*80}\nConversation {idx}/{len(web_train_data)-1}\n{'='*80}\n")
        visualize_conversation(idx)

In [ ]:
visualize_conversation(1)